# 14.2 Calculus split audit

Inspect the baseline v3 exported positive images. Near-duplicate hashes generate candidates for visual review and do not prove identity or patient leakage.

In [1]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
from collections import Counter
root = Path.cwd()
while not (root / "pyproject.toml").exists(): root = root.parent
data = root / "data/runs/calculus_v3/calculus_data"
train = sorted((data / "images/train").glob("*"))
val = sorted((data / "images/val").glob("*"))
print("train",len(train),"validation",len(val))
assert (len(train),len(val)) == (846,239), "Incomplete downloaded baseline data"

train 846 validation 239


## Candidate near-duplicates

Difference hash converts a grayscale image to a 64-bit representation. Count cross-split pairs within 4 bit differences. Similar hashes can be unrelated; altered images can be missed.

In [2]:
def dhash(path):
    image=ImageOps.grayscale(Image.open(path)).resize((9,8))
    pixels=np.asarray(image)
    return np.packbits((pixels[:,1:] > pixels[:,:-1]).flatten())
train_hash=np.stack([dhash(p) for p in train])
val_hash=np.stack([dhash(p) for p in val])
popcount=np.array([int(i).bit_count() for i in range(256)],dtype=np.uint8)
pairs=[]
for idx, value in enumerate(val_hash):
    distances=popcount[np.bitwise_xor(train_hash,value)].sum(axis=1)
    for j in np.flatnonzero(distances<=4): pairs.append((int(distances[j]),str(train[j].name),str(val[idx].name)))
pairs.sort()
print("candidate pairs",len(pairs),"validation images",len({p[2] for p in pairs}))
print("first 10 candidates:")
for p in pairs[:10]: print(p)

candidate pairs 186 validation images 98
first 10 candidates:
(0, 'calculus_-1-_jpg.rf.dd05dc49c98e19ab5dd2c26a2014232a.jpg', 'calculus_-1153-_jpg.rf.c9bef2f4db286b8a8464352fc538a7e5.jpg')
(0, 'calculus_-1026-_jpg.rf.2f8c9e2b25c83dc57f612d8b91dc44f1.jpg', 'calculus_oral-disease.v1i.yolov8_calculus-1118-_jpg.rf.339b508f7034e51b9f3d678fb552de1c.jpg')
(0, 'calculus_-105-_jpg.rf.9763ea5f4c4996144ce73967873a58d4.jpg', 'calculus_oral-disease.v1i.yolov8_calculus-1158-_jpg.rf.f9662eea873dfa8f9123b66108b214eb.jpg')
(0, 'calculus_-1156-_jpg.rf.cf9219c61f0eee4954201fe382e158ba.jpg', 'calculus_-148-_jpg.rf.caf56df14466b2a75576d0852d9654f8.jpg')
(0, 'calculus_-1156-_jpg.rf.cf9219c61f0eee4954201fe382e158ba.jpg', 'calculus_-292-_jpg.rf.4a68f915c31422c3d34ba9c08a80d550.jpg')
(0, 'calculus_-1170-_jpg.rf.44f004611ad0304b93a5639dd01a20d2.jpg', 'calculus_171_jpg.rf.c2df6d5d219b86140638073097330571.jpg')
(0, 'calculus_-1176-_jpg.rf.13d0f4ef8e94f6fa30b73c6fc0480f49.jpg', 'calculus_-1176-_jpg.rf.3acc53579286

## Box size distribution

Compute label widths and heights in 640-pixel model input equivalents. Values are normalized to the source image and do not account for letterbox padding.

In [3]:
sizes=[]
for p in (data / "labels/train").glob("*.txt"):
    for line in p.read_text().splitlines():
        _,x,y,w,h=map(float,line.split()); sizes.append((w*640,h*640))
sizes=np.asarray(sizes)
print("boxes",len(sizes))
for i,name in enumerate(["width","height"]): print(name, "10/50/90 percentiles",np.percentile(sizes[:,i],[10,50,90]))
print("min side below 32 pixels:",int((sizes.min(axis=1)<32).sum()))

boxes 5898
width 10/50/90 percentiles [ 27.18954248  54.37908497 105.        ]
height 10/50/90 percentiles [ 58.85057471 111.92118227 196.50757576]
min side below 32 pixels: 985
